***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 5 章：成像](5_0_introduction.ipynb)
    * 上一节：[5.2 采样函数与点扩散函数](5_2_sampling_functions_and_psfs.ipynb)
    * 下一节：[5.4 脏图像与可见度权重](5_4_imaging_weights.ipynb)

***


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML
import AA_filter
from convolutional_degridder import fft_degrid
from convolutional_gridder import grid_ifft
%matplotlib inline
HTML('../style/course.css') #apply general CSS


导入本节所需的专用模块。


In [ ]:
HTML('../style/code_toggle.html')


## 5.3 用于 FFT 的网格化与反网格化<a id='imaging:sec:gridding'></a>

第 5.2 节已经把脏图像写成了采样函数与真实天空之间的卷积结果。理论上，只要对每个不规则 `uv` 样本逐点做直接 Fourier 求和，就可以完成成像；但现代干涉阵的采样点数巨大，直接求和的代价很快就会变得无法接受。于是，实际成像几乎总要绕一个弯：先把不规则采样重采样到规则网格上，再利用 FFT 高效地完成 Fourier 变换。

这个“绕弯”并不是免费的。网格化会引入插值误差，FFT 的周期边界条件会带来混叠，卷积核的选择又会影响幅度精度和旁瓣污染。因此，本节的重点不是把 gridding 当成一条程序命令，而是说明它为什么必要、误差从哪里来，以及怎样通过卷积核和过采样控制这些误差。


In [ ]:
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)


def normalize(arr):
    arr = np.asarray(arr, dtype=float)
    maxval = np.max(np.abs(arr))
    return arr if maxval == 0 else arr / maxval


### 5.3.1 为什么不能直接把可见度送进 FFT

若不考虑计算代价，成像最直接的表达式是对每个图像像素显式求和：

$$
I_D(l,m)=\sum_{k=1}^{M} V_k\,e^{2\pi i(u_k l + v_k m)},
$$

其中 $M$ 是可见度样本数。这个公式没有要求 $u_k,v_k$ 落在规则网格上，因此它对真实干涉测量最忠实；但若要在 $N_l\times N_m$ 个像素上逐点计算，复杂度大约随 $N_lN_mM$ 增长。对于现代阵列，$M$ 可以轻易达到数百万乃至更多，这样的直接求和即使在并行硬件上也很昂贵。

FFT 的优势来自它只适用于规则采样格点，却能把复杂度压到近似 $N\log N$ 的量级。代价是，所有输入样本必须先落到规则的 `uv` 格点上。换句话说，FFT 的快速并不是“无条件的数学奇迹”，而是建立在规则格点这一额外结构之上的。网格化所做的工作，就是把真实的、不规则的测量样本近似嵌入到这套规则结构里。


In [ ]:
grid_sizes = np.array([64, 128, 256, 512, 1024, 2048, 4096], dtype=float)
fft_cost = grid_sizes**2 * np.log2(grid_sizes**2)
dft_cost = grid_sizes**4
fft_cost /= fft_cost[0]
dft_cost /= dft_cost[0]

fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.loglog(grid_sizes, fft_cost, 'o-', lw=2, label='FFT-like scaling')
ax.loglog(grid_sizes, dft_cost, 'o-', lw=2, label='direct Fourier summation')
ax.set_xlabel('linear image size $N$')
ax.set_ylabel('relative operation count')
ax.set_title('Why gridding is computationally attractive')
ax.grid(alpha=0.25, which='both')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fft_vs_dft_scaling.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![FFT 与 DFT 的复杂度对比](figures/fft_vs_dft_scaling.png)

**图 5.3.1** 理想化的复杂度比较。常数因子和实现细节并未纳入，但趋势非常清楚：随着图像线性尺寸增大，直接 Fourier 求和的代价增长远快于 FFT。网格化存在的根本理由，就是把问题从“不规则样本上的昂贵求和”转换成“规则网格上的快速变换”。


### 5.3.2 图像像元大小、视场与 `uv` 网格间距

一旦决定使用 FFT，图像域和 `uv` 域的采样间距就被绑定起来了。若规则 `uv` 网格步长为 $(\Delta u, \Delta v)$，图像域的总视场大致满足

$$
\Theta_l \simeq \frac{1}{\Delta u},
\qquad
\Theta_m \simeq \frac{1}{\Delta v}.
$$

若图像大小为 $N_l\times N_m$，则像元大小近似为

$$
\Delta l \simeq \frac{1}{N_l\Delta u},
\qquad
\Delta m \simeq \frac{1}{N_m\Delta v}.
$$

另一方面，`uv` 网格能表示的最高空间频率大约出现在边界 $u_{\max}\simeq N_l\Delta u/2$ 和 $v_{\max}\simeq N_m\Delta v/2$，因此像元大小也可写成

$$
\Delta l \simeq \frac{1}{2u_{\max}},
\qquad
\Delta m \simeq \frac{1}{2v_{\max}}.
$$

这组关系决定了成像参数的物理含义。更小的像元意味着更大的 FFT 网格；更大的视场意味着更密的 `uv` 网格；若像元取得过粗，长基线信息就会折叠到无法表示的高频之外；若视场取得过大而抗混叠处理又不足，图像边缘外的源就会以周期性复制的形式泄漏回视场内部。


### 5.3.3 卷积网格化与反网格化

最常见的策略是卷积型重采样。设不规则可见度样本位于 $(u_i,v_i)$，规则网格点位于 $(a\Delta u, b\Delta v)$，卷积核为 $C$，则网格点上的值可写为

$$
V_G(a,b)=\sum_i w_i V_i\,C\!\left(\frac{a\Delta u-u_i}{\Delta u},\frac{b\Delta v-v_i}{\Delta v}\right).
$$

这里的物理图像很直观：每个可见度不是被扔进最近的单个格点，而是按卷积核的形状，向附近若干个格点“涂抹”一小片权重。支持宽度越大，插值越精确，但计算量也越高。反网格化则做相反的事情：从规则 FFT 网格中，用同样或相近的核在某个不规则位置上收集样本，以预测该位置的可见度。

卷积核之所以要过采样，是因为真实样本通常不会恰好落在网格中心。若只保存“恰在格点上”的核值，就无法准确表示分数格点偏移。过采样表本质上是在核的连续曲线下预先存一张更密的查找表，以便在实际 gridding/degridding 时快速选择最接近的核系数。


![卷积网格化示意](figures/gridding_illustration.png)

**图 5.3.2** 卷积型网格化与反网格化的几何图像。网格化时，不规则样本把权重分配到附近规则格点；反网格化时，则从规则格点收集权重来预测某个不规则位置上的可见度。

![过采样卷积核示意](figures/oversampled_filter_illustration.png)

**图 5.3.3** 过采样卷积核的查找表。因为真实 `uv` 位置通常带有分数格点偏移，所以同一个支持宽度下需要为不同的分数偏移预存不同系数。过采样因子越高，分数偏移近似越精细，但内存和计算代价也越高。


### 5.3.4 混叠与抗混叠卷积核

FFT 默认输入数据在边界上周期延拓。若网格化时只是简单地把样本丢给最近邻格点，相当于在 `uv` 域使用了一个盒函数核；它在图像域中的 Fourier 对应会有缓慢衰减的旁瓣，于是视场外的源很容易以周期复制的方式泄漏回视场内部。这就是 gridding 误差中最常见的一类混叠。

因此，卷积核的任务不只是“把样本放上网格”，还要尽量把其 Fourier 响应限制在有效视场内。理想核应该在 `uv` 域足够局域，以控制计算量；在图像域又要足够快地衰减，以压低混叠。最近邻 `box` 核几乎只满足第一条，而截断 `sinc`、加窗 `sinc`、prolate spheroidal 等核，正是为兼顾这两方面而设计的。


![最近邻插值的混叠](figures/NN_interpolation_aliasing.png)

**图 5.3.4** 最近邻累积（盒函数核）产生的显著混叠。由于核在图像域中的旁瓣衰减缓慢，视场外的源会折叠回视场内。

![抗混叠卷积核的改进](figures/AA_kernel_alias_reduction.png)

**图 5.3.5** 采用更平滑的抗混叠核后，视场外复制伪影被明显压低。这就是现代卷积型 gridding 普遍不再使用最近邻核的原因。


In [ ]:
kernels = {
    'Box': AA_filter.AA_filter(1, 63, 'box'),
    'Sinc': AA_filter.AA_filter(3, 63, 'sinc'),
    'Gaussian-sinc': AA_filter.AA_filter(3, 63, 'gaussian_sinc'),
}

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
for name, kernel in kernels.items():
    taps = np.arange(kernel.no_taps) / float(kernel.oversample) - kernel.full_sup / 2
    spectrum = np.abs(np.fft.fftshift(np.fft.fft(np.fft.ifftshift(kernel.filter_taps))))
    spectrum = spectrum / np.max(spectrum)
    freq = np.linspace(-1.0, 1.0, spectrum.size)
    axes[0].plot(taps, kernel.filter_taps, lw=2, label=name)
    axes[1].plot(freq[freq >= 0], 20.0 * np.log10(spectrum[freq >= 0] + 1e-8), lw=2, label=name)

axes[0].set_title('convolution kernels in uv space')
axes[0].set_xlabel('fractional grid offset')
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].set_title('magnitude of Fourier responses')
axes[1].set_xlabel('normalized image-space frequency')
axes[1].set_ylabel('dB')
axes[1].set_ylim(-80, 5)
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

fig.tight_layout()
fig.savefig(FIG_DIR / 'gridding_kernel_spectra.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![卷积核与其 Fourier 响应](figures/gridding_kernel_spectra.png)

**图 5.3.6** 盒函数、截断 `sinc` 和 Gaussian-sinc 卷积核的实空间形状及其 Fourier 响应。盒函数最局域，但频域旁瓣最高；更平滑的核在图像域中能更快压低旁瓣，从而更有效地抑制混叠。


### 5.3.5 一个完整示例：从模型天空到脏图像

下面用一个合成天空和一组不规则 `uv` 采样，把前面的概念串起来。先从模型图像出发，通过反网格化在不规则 `uv` 位置上预测可见度；再用不同卷积核把这些样本网格化回规则平面，并做逆 FFT 生成脏图。这样可以直接比较卷积核选择如何影响成像残差。


In [ ]:
np.random.seed(3)

demo_N = 96
demo_field = 1.2
demo_cell = demo_field / demo_N
demo_l = (np.arange(demo_N) - demo_N / 2) * demo_cell
demo_m = (np.arange(demo_N) - demo_N / 2) * demo_cell
demo_ll, demo_mm = np.meshgrid(demo_l, demo_m)

demo_sky = (
    1.00 * np.exp(-((demo_ll + 0.18) ** 2 + (demo_mm - 0.10) ** 2) / (2 * 0.03 ** 2))
    + 0.55 * np.exp(-((demo_ll - 0.12) ** 2 + (demo_mm + 0.16) ** 2) / (2 * 0.06 ** 2))
)
for l0, m0, flux in [(0.24, -0.22, 0.35), (-0.30, 0.28, 0.22)]:
    ix = np.argmin(np.abs(demo_l - l0))
    iy = np.argmin(np.abs(demo_m - m0))
    demo_sky[iy, ix] += flux

demo_pairs = 260
demo_radius = 0.34 * demo_N
demo_angles = np.random.uniform(0.0, 2.0 * np.pi, demo_pairs)
demo_radii = demo_radius * np.sqrt(np.random.uniform(0.03, 1.0, demo_pairs))
demo_half = np.column_stack([
    demo_radii * np.cos(demo_angles),
    demo_radii * np.sin(demo_angles),
    np.zeros(demo_pairs),
])
demo_uv = np.vstack([demo_half, -demo_half])
ref_lda = np.array([1.0])

kernel_bank = {
    'Box': AA_filter.AA_filter(1, 63, 'box'),
    'Sinc': AA_filter.AA_filter(3, 63, 'sinc'),
    'Gaussian-sinc': AA_filter.AA_filter(3, 63, 'gaussian_sinc'),
}

results = {}
for name, kernel in kernel_bank.items():
    vis = fft_degrid(demo_sky.reshape(1, demo_N, demo_N), demo_uv, ref_lda, demo_N, demo_N, kernel)
    dirty, psf = grid_ifft(vis, demo_uv, ref_lda, demo_N, demo_N, kernel)
    psf_peak = np.max(np.real(psf))
    dirty_image = np.real(dirty[0]) / psf_peak
    residual = dirty_image - demo_sky
    results[name] = {
        'dirty': dirty_image,
        'residual': residual,
        'rms': np.sqrt(np.mean(residual ** 2)),
    }

extent = [demo_l[0], demo_l[-1], demo_m[0], demo_m[-1]]
residual_scale = max(np.max(np.abs(result['residual'])) for result in results.values())

fig, axes = plt.subplots(2, 5, figsize=(16.0, 7.8))
axes[0, 0].imshow(demo_sky, cmap='gray', origin='lower', extent=extent)
axes[0, 0].set_title('model sky')
axes[0, 0].set_xlabel('l (deg)')
axes[0, 0].set_ylabel('m (deg)')

axes[1, 0].scatter(demo_uv[:, 0], demo_uv[:, 1], s=4, c='k')
axes[1, 0].set_title('irregular uv samples')
axes[1, 0].set_xlabel('u (grid units)')
axes[1, 0].set_ylabel('v (grid units)')
axes[1, 0].set_aspect('equal')

axes[0, 1].axis('off')
axes[1, 1].axis('off')

for col, name in enumerate(results, start=2):
    axes[0, col].imshow(results[name]['dirty'], cmap='gray', origin='lower', extent=extent)
    axes[0, col].set_title(f"{name} dirty image")
    axes[0, col].set_xlabel('l (deg)')
    axes[0, col].set_ylabel('m (deg)')

    axes[1, col].imshow(
        results[name]['residual'],
        cmap='coolwarm',
        origin='lower',
        extent=extent,
        vmin=-residual_scale,
        vmax=residual_scale,
    )
    axes[1, col].set_title(f"{name} residual\nRMS={results[name]['rms']:.4f}")
    axes[1, col].set_xlabel('l (deg)')
    axes[1, col].set_ylabel('m (deg)')

fig.tight_layout()
fig.savefig(FIG_DIR / 'gridding_kernel_comparison.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![不同卷积核的成像对比](figures/gridding_kernel_comparison.png)

**图 5.3.7** 同一组合成天空和不规则 `uv` 采样，在不同卷积核下得到的脏图与残差。最近邻 `Box` 核最快，但残差也最明显；`Sinc` 与 `Gaussian-sinc` 在相同采样条件下能更好地控制混叠和插值误差，因此重建结果更接近原始模型。

这个例子也说明了一件常被忽略的事：gridding 不是一个完全中性的数值步骤。卷积核、支持宽度和过采样因子都会把自己的误差结构带进图像。因此，后面讨论的权重函数和去卷积，并不是在一个“无误差的 FFT 图像”上继续加工，而是在一个已经包含 gridding 近似的脏图像上继续工作。


### 5.3.6 小结

本节的逻辑可以概括为三句话。第一，FFT 之所以快，是因为它要求规则格点；真实干涉可见度之所以不能直接送进 FFT，是因为采样天然不规则。第二，gridding/degridding 用卷积核在不规则样本与规则网格之间搭桥，从而把物理测量转换成可高效计算的离散 Fourier 问题。第三，卷积核既决定插值精度，也决定混叠控制能力，因此它本身就是成像误差预算的一部分。

下一节将在此基础上继续前进。即便卷积核固定不变，样本如何加权仍会改变脏波束主瓣宽度、旁瓣高度以及图像噪声分布。也就是说，走出 gridding 之后，仍然要面对一个新的设计自由度：可见度权重。


***

* 下一节：[5.4 脏图像与可见度权重](5_4_imaging_weights.ipynb)
